In [1]:
import pandas as pd
import openai
from dotenv import load_dotenv
from openai import OpenAI
import os
import json

In [2]:
#setup OpenAI API
load_dotenv()
#client = OpenAI()

True

In [3]:
def call_openai_api(prompt, model):
    client = OpenAI()

    if model == 'o1-mini':
        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]
    else:
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": prompt
            }
        ]

    completion = client.chat.completions.create(
    model=model,
    messages=messages
)
    return completion

In [32]:
QUERY = f"""
Search for comprehensive diabetes datasets that include long-term glycemic monitoring parameters, patient demographics, associated complications,
 and treatment outcomes. I'm looking for datasets with at least 5 years of clinical follow-up, with particular attention to HbA1c levels, 
 pharmacological therapy used, and cardiovascular comorbidities. Preferably, the datasets should include lifestyle indicators such as physical 
 activity and dietary habits, and distinguish between type 1 and type 2 diabetes. """

In [33]:
prompt_to_topic = f""" 
Analyze the following query:
{QUERY}
Understand which is the main topic of the query.

An example is:
Query: "I want to find datasets about Parkinson and its impact on the human mind, especially I need data about age of the patients."
Output: "Parkinson"

The output should be composed of the topic alone, no additional text is accepted nor required.
"""

In [6]:
topic = call_openai_api(prompt_to_topic, 'o1-mini').choices[0].message.content

In [7]:
topic

'Diabetes'

In [8]:
summary = f"""Diabetes is a complex metabolic disorder characterized by chronic high blood sugar levels, resulting from either insufficient insulin production or the body's inability to effectively use insulin. 
Several key factors contribute to the development and progression of diabetes, including demographic, physiological, and genetic influences. 
Demographic factors such as age and pregnancy history can play a role in diabetes risk, as older individuals and women with a history of gestational diabetes may have an increased likelihood of developing the condition. 
Physiological aspects are central to diabetes diagnosis and management, with critical health metrics including glucose levels, blood pressure, insulin sensitivity, skin thickness, and Body Mass Index (BMI). 
These measurements provide valuable insights into an individual's metabolic health and cardiovascular function, helping to identify potential warning signs of diabetes. 
Additionally, genetic predisposition plays a significant role, with hereditary factors influencing susceptibility to the disease. 
The interplay between these elements determines an individual's overall risk, shaping both prevention strategies and treatment approaches. 
Understanding diabetes from this multifaceted perspective enables more effective interventions, personalized care plans, and advancements in medical research aimed at mitigating its impact on global health."""


In [9]:
RESPONSE_QUERIES = """ 
{
  "queries": [
    {
      "query": "Select data ..."
    },
    {
      "query": "Find datasets ..."
    },
    {
      "query": "Show me data ..."
    }
  ]
}
"""

In [37]:
final_prompt = f"""
You are an AI assistant specialized in optimizing complex prompt instructions. You will be given a main instruction, a general topic, and a brief summary of the topic. Your task is to break down the main instruction into smaller, more specific instructions while strictly maintaining the context of the original instruction.
The instructions are used to find datasets in a dataspace, using an advanced RAG system in which they will be matched with questions such as "Find dataset that includes information X".

Input:
Main instruction: A complex instruction that needs to be divided into sub-instructions.
General topic: The main subject of the instruction.
Topic summary: A short description that helps clarify the context of the instruction but should not be used to introduce additional subqueries outside the scope of the original instruction.

Expected output:
Provide a list of sub-instructions that:
1. Maintain the same structure as the original instruction
2. Cover all key aspects mentioned in the complex instruction
3. Are specific and actionable
4. Collectively address the full scope of the original instruction
7. Ensure that each sub-instruction is directly derived from the main instruction and does not introduce new elements not explicitly mentioned
8. Use the topic summary only to understand the context of the instruction, not to generate new sub-instructions
9. If the main instruction includes specific constraints or priorities (e.g., 'at least 5 years of follow-up'), ensure these are reflected in the relevant sub-instructions

Example input/output:
Input:

Main instruction: "Find datasets containing comprehensive obesity information that include long-term weight fluctuations, metabolic parameters,
 comorbidity development, and intervention outcomes. I'm looking for datasets with at least 5 years of follow-up, with particular attention to 
 BMI changes, dietary patterns, and physical activity levels. Preferably, the datasets should include psychological factors, socioeconomic 
 determinants, and differentiate between childhood and adult-onset obesity."

General topic: "Obesity"

Topic summary: "Obesity is a complex and multifactorial health condition characterized by excessive body fat accumulation, which increases the risk of various chronic diseases. 
Several key factors contribute to the development and progression of obesity, including demographic, physiological, genetic, and lifestyle-related influences. 
Demographic aspects such as age, gender, and socioeconomic status can impact obesity prevalence, as certain populations may be more vulnerable due to environmental and behavioral factors. 
Physiological markers, including Body Mass Index (BMI), body fat percentage, blood pressure, and metabolic indicators, play a crucial role in assessing an individual's overall health and obesity risk. 
Additionally, genetic predisposition influences susceptibility, with hereditary factors contributing to variations in metabolism, fat storage, and appetite regulation. 
Lifestyle choices such as diet composition, physical activity levels, and sedentary behaviors are central to obesity prevention and management. 
The interaction of these elements determines an individual's likelihood of developing obesity, shaping both public health strategies and personalized intervention approaches. 
A comprehensive understanding of obesity through these dimensions is essential for developing effective prevention, treatment, and policy measures to mitigate its impact on global health."

Output:

[
    "Find datasets containing comprehensive obesity information",
    "Find obesity datasets with long-term weight fluctuations",
    "Find datasets with metabolic parameters for obese patients",
    "Find datasets with comorbidity development for obese patients",
    "Find datasets with intervention outcomes for obese patients",
    "Find obesity datasets with at least 5 years of follow-up",
    "Find obesity datasets with BMI changes",
    "Find datasets with dietary patterns for obese patients",
    "Find datasets with physical activity levels of obese patients",
    "Find obesity datasets with psychological factors",
    "Find obesity datasets with socioeconomic determinants",
    "Find datasets differentiating between childhood and adult-onset obesity"
]

Work with this data:
Main instruction: "{QUERY}"
General topic: "{topic}"
Topic summary: "{summary}"

Provide the answer as a JSON object with no additional context or formatting.
Example: {RESPONSE_QUERIES}

"""

In [38]:
subqueries = call_openai_api(final_prompt, 'gpt-4o-mini').choices[0].message.content

In [39]:
subqueries = subqueries.replace("```json\n", "").replace("\n```", "")

subqueries_dict = json.loads(subqueries)

From the original query we expect something like:
1. long-term glycemic parameters
2. patient demographics
2. associated complications
3. treatment outcomes
4. 5 years of clinical follow-up
5. HbA1c levels
6. pharmacological therapy used 
7. cardiovascular comorbidities
8. physical activity and dietary habits
9. type 1 and type 2 diabetes

In [40]:
for i, query in enumerate(subqueries_dict['queries'], start=1):
    print(f"{i}. {query['query']}\n")

1. Search for comprehensive diabetes datasets

2. Find diabetes datasets that include long-term glycemic monitoring parameters

3. Find diabetes datasets with patient demographics

4. Find diabetes datasets that include associated complications

5. Find diabetes datasets that report treatment outcomes

6. Locate diabetes datasets with at least 5 years of clinical follow-up

7. Search for diabetes datasets with HbA1c levels

8. Find diabetes datasets detailing pharmacological therapy used

9. Find diabetes datasets with cardiovascular comorbidities

10. Search for diabetes datasets that include lifestyle indicators such as physical activity

11. Find diabetes datasets that include dietary habits

12. Find diabetes datasets that distinguish between type 1 and type 2 diabetes



Proviamo con un approccio diverso

In [10]:
RESPONSE_KEYS = """"
{
  "keys": [
    {"key_point":"key point 1"},
    {"key_point":"key point 2"},
  ]
}
"""

In [36]:
prompt_key_points = f"""    
Analyze the following text and identify the key elements for dataset research.

Extraction criteria:
- Identify the main concepts
- Detect any specific parameters
- Capture contextual details
- Extract selection constraints or criteria


Input: {QUERY}

Output: 
[List of extracted key elements]

Provide the answer as a JSON object with no additional context or formatting, maintaining the order of relevance.
Example:{RESPONSE_KEYS}

"""

In [13]:
key_points = call_openai_api(prompt_key_points, 'gpt-4o-mini').choices[0].message.content

In [14]:
key_points = key_points.replace("```json\n", "").replace("\n```", "")

key_points_dict = json.loads(key_points)

for i, key in enumerate(key_points_dict['keys'], start=1):
    print(f"{i}. {key['key_point']}\n")

1. diabetes datasets

2. long-term glycemic monitoring parameters

3. patient demographics

4. associated complications

5. treatment outcomes

6. 5 years of clinical follow-up

7. HbA1c levels

8. pharmacological therapy used

9. cardiovascular comorbidities

10. lifestyle indicators

11. physical activity

12. dietary habits

13. type 1 diabetes

14. type 2 diabetes



In [35]:
prompt_KP_query = f"""
I have a complex query and a set of extracted key points. I want to break down this query into precise, targeted subqueries that comprehensively address each key point.

Please help me by:

    Reviewing the complex query and the extracted key points
    Analyzing the relationship and coverage between the key points
    Generating a set of specific, focused subqueries that:
        Directly address each key point
        Ensure no important aspect of the original query is missed
        Are clear, concise, and actionable
        Can be potentially answered independently

Please provide the subqueries in a numbered list, with a brief explanation of how each subquery relates to the original query and covers its respective key point.

Complex Query: {QUERY}
Key Points: {key_points_dict}"

The output should be a JSON object with the subqueries like this:
{RESPONSE_QUERIES}
No additional text or formatting is required.
"""

In [22]:
subqueries_with_KP = call_openai_api(prompt_KP_query, 'gpt-4o-mini').choices[0].message.content

In [23]:
subqueries_with_KP

'{\n  "queries": [\n    {\n      "query": "Find comprehensive diabetes datasets that include long-term glycemic monitoring parameters."\n    },\n    {\n      "query": "Look for diabetes datasets that provide detailed patient demographics."\n    },\n    {\n      "query": "Identify datasets that include associated complications related to diabetes."\n    },\n    {\n      "query": "Search for datasets that report treatment outcomes for diabetes."\n    },\n    {\n      "query": "Locate diabetes datasets that have at least 5 years of clinical follow-up."\n    },\n    {\n      "query": "Retrieve diabetes datasets that specifically include HbA1c levels."\n    },\n    {\n      "query": "Find datasets detailing pharmacological therapy used in diabetes management."\n    },\n    {\n      "query": "Search for diabetes datasets that include information on cardiovascular comorbidities."\n    },\n    {\n      "query": "Identify datasets that contain lifestyle indicators, such as physical activity and

In [25]:
subqueries_with_KP = subqueries_with_KP.replace("```json\n", "").replace("\n```", "")

subqueries_with_KP_dict = json.loads(subqueries_with_KP)

for i, key in enumerate(subqueries_with_KP_dict['queries'], start=1):
    print(f"{i}. {key['query']}\n")

1. Find comprehensive diabetes datasets that include long-term glycemic monitoring parameters.

2. Look for diabetes datasets that provide detailed patient demographics.

3. Identify datasets that include associated complications related to diabetes.

4. Search for datasets that report treatment outcomes for diabetes.

5. Locate diabetes datasets that have at least 5 years of clinical follow-up.

6. Retrieve diabetes datasets that specifically include HbA1c levels.

7. Find datasets detailing pharmacological therapy used in diabetes management.

8. Search for diabetes datasets that include information on cardiovascular comorbidities.

9. Identify datasets that contain lifestyle indicators, such as physical activity and dietary habits.

10. Locate datasets that distinguish between type 1 and type 2 diabetes.

11. Find datasets that contain information on physical activity levels of diabetes patients.

12. Search for diabetes datasets that include information on dietary habits of patient

An idea is to use both key points and summary. Summary can be helpful especially when the topic is not famous.

In [42]:
prompt_summary_kp = f"""
You are an AI assistant specialized in optimizing complex prompt instructions. You will be given a main instruction, a general topic, a brief summary of the topic and the key points of the query. Your task is to break down the main instruction into smaller, more specific instructions while strictly maintaining the context of the original instruction.
The instructions are used to find datasets in a dataspace, using an advanced RAG system in which they will be matched with questions such as "Find dataset that includes information X".

Input:
Main instruction: A complex instruction that needs to be divided into sub-instructions.
General topic: The main subject of the instruction.
Topic summary: A short description that helps clarify the context of the instruction but should not be used to introduce additional subqueries outside the scope of the original instruction.

Expected output:
Provide a list of sub-instructions that:
1. Maintain the same structure as the original instruction
2. Cover all key aspects mentioned in the complex instruction
3. Are specific and actionable
4. Collectively address the full scope of the original instruction
7. Ensure that each sub-instruction is directly derived from the main instruction and does not introduce new elements not explicitly mentioned
8. Use the topic summary only to understand the context of the instruction, not to generate new sub-instructions
9. If the main instruction includes specific constraints or priorities (e.g., 'at least 5 years of follow-up'), ensure these are reflected in the relevant sub-instructions

Example input/output:
Input:

Main instruction: "Find datasets containing comprehensive obesity information that include long-term weight fluctuations, metabolic parameters,
 comorbidity development, and intervention outcomes. I'm looking for datasets with at least 5 years of follow-up, with particular attention to 
 BMI changes, dietary patterns, and physical activity levels. Preferably, the datasets should include psychological factors, socioeconomic 
 determinants, and differentiate between childhood and adult-onset obesity."

General topic: "Obesity"

Topic summary: "Obesity is a complex and multifactorial health condition characterized by excessive body fat accumulation, which increases the risk of various chronic diseases. 
Several key factors contribute to the development and progression of obesity, including demographic, physiological, genetic, and lifestyle-related influences. 
Demographic aspects such as age, gender, and socioeconomic status can impact obesity prevalence, as certain populations may be more vulnerable due to environmental and behavioral factors. 
Physiological markers, including Body Mass Index (BMI), body fat percentage, blood pressure, and metabolic indicators, play a crucial role in assessing an individual's overall health and obesity risk. 
Additionally, genetic predisposition influences susceptibility, with hereditary factors contributing to variations in metabolism, fat storage, and appetite regulation. 
Lifestyle choices such as diet composition, physical activity levels, and sedentary behaviors are central to obesity prevention and management. 
The interaction of these elements determines an individual's likelihood of developing obesity, shaping both public health strategies and personalized intervention approaches. 
A comprehensive understanding of obesity through these dimensions is essential for developing effective prevention, treatment, and policy measures to mitigate its impact on global health."

Key Points: [
1. Minimum 5 years of longitudinal follow-up
2. Long-term weight fluctuations tracking
3. BMI changes measurement
4. Metabolic parameters monitoring
5. Dietary patterns documentation
6. Physical activity levels tracking
7. Intervention outcomes assessment
8. Psychological factors inclusion
9. Socioeconomic determinants analysis
10. Differentiation between childhood and adult obesity onset
11. Comorbidity development tracking
12. Comprehensive obesity-related information]

Output:

[
    "Find datasets containing comprehensive obesity information",
    "Find obesity datasets with long-term weight fluctuations",
    "Find datasets with metabolic parameters for obese patients",
    "Find datasets with comorbidity development for obese patients",
    "Find datasets with intervention outcomes for obese patients",
    "Find obesity datasets with at least 5 years of follow-up",
    "Find obesity datasets with BMI changes",
    "Find datasets with dietary patterns for obese patients",
    "Find datasets with physical activity levels of obese patients",
    "Find obesity datasets with psychological factors",
    "Find obesity datasets with socioeconomic determinants",
    "Find datasets differentiating between childhood and adult-onset obesity"
]

Work with this data:
Main instruction: "{QUERY}"
General topic: "{topic}"
Topic summary: "{summary}"
Key points: "{key_points_dict}"

Provide the answer as a JSON object with no additional context or formatting.
Example: {RESPONSE_QUERIES}

"""

In [43]:
subqueries_summary_KP = call_openai_api(prompt_summary_kp, 'gpt-4o-mini').choices[0].message.content

subqueries_summary_KP = subqueries_summary_KP.replace("```json\n", "").replace("\n```", "")

subqueries_summary_KP_dict = json.loads(subqueries_summary_KP)

for i, key in enumerate(subqueries_summary_KP_dict['queries'], start=1):
    print(f"{i}. {key['query']}\n")

1. Search for comprehensive diabetes datasets

2. Find diabetes datasets including long-term glycemic monitoring parameters

3. Locate datasets with patient demographics for diabetes

4. Identify diabetes datasets with associated complications

5. Gather diabetes datasets with treatment outcomes

6. Find diabetes datasets with at least 5 years of clinical follow-up

7. Search for datasets with HbA1c levels in diabetes patients

8. Locate diabetes datasets documenting pharmacological therapy used

9. Identify datasets with cardiovascular comorbidities in diabetes patients

10. Gather diabetes datasets including lifestyle indicators

11. Find diabetes datasets with physical activity information

12. Search for diabetes datasets with dietary habits information

13. Identify datasets differentiating between type 1 and type 2 diabetes

